# 005 — уверенные эпизоды и восстановление спорных промежутков
**Run All** запускает перебор 243 конфигураций и сохраняет полный прогноз. tqdm показывает прогресс.

Основное правило: **A → неопределённость → A означает A на всём промежутке**. То же для H. Правило действует независимо от длины промежутка, но никогда не пересекает D или перерыв. Между разными командами допускается ровно одна граница; сравниваем середину, средневзвешенный кадр и максимум согласованности. Это ограничение может пропустить реальные короткие обратные перехваты.

Это офлайн-метод: использует будущие наблюдения для поиска уверенного эпизода. **Не подглядывает в будущие истинные метки**. Уверенность геометрическая, а не доказательство контроля мяча.

Уверенный кадр: мяч наблюдается; видно минимум 3 точки каждой команды; ближайший игрок своей команды не дальше radius, соперник не ближе radius+margin. Подтверждаем непрерывной серией min_confident_frames; всю серию принимаем ретроспективно. Спорные отдельные короткие серии остаются неопределённостью. Номера треков не используются как личности.

Внутри спорного интервала свидетельства за команды — exp(−0.5×(ближайшее расстояние/radius)²), с нулём при недостаточных наблюдениях. При необходимости сглаживаем их симметрично внутри игрового интервала. Это оценки, не калиброванные вероятности.

**Граница:** середина — контроль; weighted — среднее кадров с весом роста преимущества новой команды плюс 0.1×близость обеих команд; consistent — максимальная сумма поддержки старой команды слева и новой справа. При полном отсутствии свидетельств — середина. При равных максимумах consistent выбирает ближайший к середине.

Незамкнутые края: ближайший якорь распространяется максимум на 2 секунды; дальше X. Если якорей нет, весь живой интервал X. D — неизменённый детектор 001.

Параметры выбираются только на первых четырёх минутах (spellF1, затем macroF1, затем номер варианта). Последняя минута — проверка. **Якоря и сглаживание не пересекают границу train/validation**. Полный прогноз матча использует весь доступный контекст и потому может отличаться на краях от изолированной проверки.

Всего 17 публичных владений; 243 варианта легко переобучаются. Проверочная минута уже использовалась в проекте и не является независимым тестом. После просмотра проверки параметры не меняем.
Все результаты, таблицы, подробные границы и прогнозы остаются в local_review/experiments/005_confident_anchors, вне Git. Предыдущие эксперименты не изменяются.


In [ ]:
from pathlib import Path
import sys, json, time, warnings, hashlib, itertools, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
sys.path.insert(0,str(ROOT))
from games import GAME, GAMES, asset_paths, half_video_blocks
from starter.baseline import team_of_each_track
from starter.load_smart_tagging import load as load_smart_tagging
from evaluation import possession_metrics as pm
EXPERIMENT = "005_confident_anchors"
OUT = ROOT / "local_review/experiments" / EXPERIMENT
OUT.mkdir(parents=True,exist_ok=True)
ignore = ROOT / "local_review/.gitignore"
if not ignore.exists(): ignore.write_text("*\n")
CFG = json.loads((ROOT / "sol/001_box5_vote100/config.json").read_text())
GRID = json.loads((ROOT / "sol" / EXPERIMENT / "grid.json").read_text())
N, FPS = GAMES[GAME]["n_video_frames"], GAMES[GAME]["fps"]
PITCH = np.array(GAMES[GAME]["pitch"],dtype=np.float64)
BLOCKS = half_video_blocks(GAME)
PATHS = asset_paths()

def save_json(path, value):
    temp = path.with_suffix(path.suffix+".tmp")
    temp.write_text(json.dumps(value,indent=2,ensure_ascii=False))
    temp.replace(path)

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

started=time.perf_counter()
print(EXPERIMENT, "| Results:", OUT)


## Данные и общий детектор D

In [ ]:
# Load only ATD positions; raw speed units are inconsistent in the source documentation.
# Читаем только координаты ATD: единицы готовой скорости неоднозначны.
track_teams = team_of_each_track(PATHS["atd_meta"])
players = np.full((N, len(track_teams), 2), np.nan, dtype=np.float64)
ball = np.full((N, 2), np.nan, dtype=np.float64)
read_count = 0
with open(PATHS["atd_raw"]) as handle:
    for i, line in enumerate(tqdm(handle,total=N,desc="Tracking / Трекинг")):
        frame_id, player_text, ball_text = line.strip().split(":")
        assert int(frame_id) == i + 1, "ATD/video alignment mismatch"
        player_values = np.fromstring(player_text.replace(";", ","), sep=",").reshape(-1, 3)
        assert len(player_values) == len(track_teams)
        players[i] = player_values[:, :2] * PITCH
        ball[i] = np.fromstring(ball_text, sep=",")[:2] * PITCH
        read_count += 1
assert read_count == N
ball_seen = np.isfinite(ball).all(axis=1)
players_seen = np.isfinite(players).all(axis=2)

# The window is an axis-aligned SQUARE, not a radius-2.5 m circle.
# Окно — КВАДРАТ вдоль осей поля, не круг радиусом 2.5 м.
def box_votes(player_xy, ball_xy, team_ids, width_m):
    inside = (np.isfinite(player_xy).all(axis=2)
              & np.isfinite(ball_xy).all(axis=1)[:, None]
              & (np.abs(player_xy - ball_xy[:, None, :]) <= width_m / 2).all(axis=2))
    home_count = (inside & (team_ids == 0)).sum(axis=1)
    away_count = (inside & (team_ids == 1)).sum(axis=1)
    # +1 means H, -1 means A, 0 means abstention for this frame.
    # +1 — H, -1 — A, 0 — нет голоса в этом кадре.
    votes = np.sign(home_count - away_count).astype(np.int8)
    return home_count, away_count, votes

home_count, away_count, frame_votes = box_votes(players, ball, track_teams, CFG["box_width_m"])
# A narrow synthetic check verifies corners, ties, and missing-ball handling.
# Проверяем на искусственных точках углы квадрата, ничью и отсутствие мяча.
test_players = np.array([[[2.5, 2.5], [2.6, 0]], [[0, 0], [0, 0]], [[0, 0], [0, 0]]])
test_ball = np.array([[0., 0.], [0., 0.], [np.nan, np.nan]])
assert box_votes(test_players, test_ball, np.array([0, 1]), 5)[2].tolist() == [1, 0, 0]

# A set-piece interval provides context only; its team tag is not used as a possession label.
# Интервал стандарта — только контекст; его Team не копируем в метку владения.
set_piece_context = np.zeros(N, dtype=bool)
for event in load_smart_tagging():
    if (event["code"] or "").strip().upper() == "SET PIECES":
        start = max(0, event["start_frame"])
        stop = min(N, event["end_frame"])
        set_piece_context[start:stop] = True
print("ATD and automatic set-piece context loaded; no ground truth accessed.")


In [ ]:
lag = CFG["motion_lag_frames"]
dt = lag / FPS
player_motion = np.full(N, np.nan)
ball_motion = np.full(N, np.nan)

# Calculate motion separately in each half, so halftime never creates a false displacement.
# Движение считаем отдельно внутри таймов, не соединяя их через перерыв.
for _, first, last in BLOCKS:
    stop = last + 1
    speed = np.linalg.norm(players[first + lag:stop] - players[first:stop - lag], axis=2) / dt
    speed[(speed > CFG["max_player_motion_mps"]) | ~np.isfinite(speed)] = np.nan
    enough = np.isfinite(speed).sum(axis=1) >= CFG["minimum_motion_tracks"]
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="All-NaN slice encountered", category=RuntimeWarning)
        medians = np.nanmedian(speed, axis=1)
    medians[~enough] = np.nan
    player_motion[first + lag:stop] = medians
    seen = ball_seen[first:stop]
    cumulative = np.r_[0, np.cumsum(seen)]
    continuous = (cumulative[lag + 1:] - cumulative[:-(lag + 1)]) == lag + 1
    ball_speed = np.linalg.norm(ball[first + lag:stop] - ball[first:stop - lag], axis=1) / dt
    ball_speed[~continuous] = np.nan
    ball_motion[first + lag:stop] = ball_speed

outside = ball_seen & ((ball < -CFG["outside_margin_m"]) | (ball > PITCH + CFG["outside_margin_m"])).any(axis=1)
inside_field = ball_seen & (ball >= 0).all(axis=1) & (ball <= PITCH).all(axis=1)

def detect_dead_ball():
    dead = np.zeros(N, dtype=bool)
    # Diagnostic counts record triggers, not ground-truth stopping reasons.
    # Счётчики показывают срабатывания эвристик, а не истинные причины остановок.
    reasons = {"outside": 0, "stationary": 0, "missing_and_slow": 0}
    for _, first, last in BLOCKS:
        state = False
        outside_run = missing_run = stationary_run = slow_run = restart_ball_run = restart_players_run = 0
        for i in range(first, last + 1):
            outside_run = outside_run + 1 if outside[i] else 0
            missing_run = missing_run + 1 if not ball_seen[i] else 0
            stationary_run = stationary_run + 1 if ball_motion[i] <= CFG["stationary_ball_mps"] else 0
            slow_run = slow_run + 1 if player_motion[i] <= CFG["slow_players_mps"] else 0
            restart_ball_run = restart_ball_run + 1 if inside_field[i] and ball_motion[i] >= CFG["restart_ball_mps"] else 0
            restart_players_run = restart_players_run + 1 if player_motion[i] >= CFG["restart_players_mps"] else 0
            if not state:
                stationary_threshold = (CFG["set_piece_stationary_confirm_frames"] if set_piece_context[i]
                                        else CFG["stationary_ball_confirm_frames"])
                slow = slow_run >= CFG["slow_players_confirm_frames"]
                if outside_run >= CFG["outside_confirm_frames"]:
                    state = True
                    reasons["outside"] += 1
                elif slow and stationary_run >= stationary_threshold:
                    state = True
                    reasons["stationary"] += 1
                elif slow and missing_run >= CFG["missing_ball_confirm_frames"]:
                    state = True
                    reasons["missing_and_slow"] += 1
            elif not outside[i] and (restart_ball_run >= CFG["restart_ball_confirm_frames"]
                                    or restart_players_run >= CFG["restart_players_confirm_frames"]):
                state = False
                # Reset evidence so a previous stoppage cannot immediately trigger a new one.
                # Сбрасываем накопленные признаки предыдущей остановки.
                outside_run = missing_run = stationary_run = slow_run = 0
            dead[i] = state
    return dead, reasons

dead, dead_trigger_counts = detect_dead_ball()


## Геометрические якоря и восстановление промежутков

In [ ]:
# Compute only aggregate geometric support; no track identity is assumed.
# Геометрические свидетельства без предположения о постоянной личности слота.
distance=np.linalg.norm(players-ball[:,None,:],axis=2)
distance[~players_seen|~ball_seen[:,None]]=np.inf
nearest=np.column_stack([distance[:,track_teams==t].min(axis=1) for t in (0,1)])
team_counts=np.column_stack([players_seen[:,track_teams==t].sum(axis=1) for t in (0,1)])
reliable=ball_seen&(team_counts>=3).all(axis=1)
del distance

def runs_of(values):
    starts=np.r_[0,np.flatnonzero(values[1:]!=values[:-1])+1]
    ends=np.r_[starts[1:],len(values)]
    return zip(starts,ends)

def smooth_evidence(values,window):
    # Offline centered smoothing, strictly inside the current live interval.
    # Офлайн-сглаживание строго внутри текущего игрового интервала.
    ids=np.arange(len(values));left=np.maximum(0,ids-window//2)
    right=np.minimum(len(values),ids+window//2+1)
    sums=np.vstack([np.zeros((1,2)),np.cumsum(values,axis=0)])
    return (sums[right]-sums[left])/(right-left)[:,None]

def choose_boundary(evidence,start,stop,old,new,method):
    # Boundary k means old team before k, new team from k onward.
    # Граница k: старая команда до k, новая начиная с k.
    middle=(start+stop)/2
    if method=='midpoint': return int(np.floor(middle))
    segment=evidence[start:stop]
    if not len(segment) or not np.any(segment): return int(np.floor(middle))
    if method=='consistent':
        old_prefix=np.r_[0.,np.cumsum(segment[:,old])]
        new_prefix=np.r_[0.,np.cumsum(segment[:,new])]
        objective=old_prefix+new_prefix[-1]-new_prefix
        candidates=np.flatnonzero(np.isclose(objective,objective.max(),atol=1e-12,rtol=0))+start
        return int(candidates[np.argmin(abs(candidates-middle))])
    if method=='weighted':
        # Rising support for the new team plus a small contest-proximity term.
        # Рост преимущества новой команды плюс небольшой вес близости обеих команд.
        contrast=evidence[:,new]-evidence[:,old]
        candidates=np.arange(start,stop+1)
        now=np.minimum(candidates,len(evidence)-1)
        prev=np.maximum(candidates-1,0)
        rise=np.maximum(contrast[now]-contrast[prev],0.)
        contested=np.minimum(evidence[now,old],evidence[now,new])
        weight=rise+.1*contested
        if weight.sum()<=1e-12:return int(np.floor(middle))
        return int(np.clip(np.floor(np.dot(candidates,weight)/weight.sum()+.5),start,stop))
    raise ValueError(method)

def fill_live(raw,evidence,min_frames,method,edge_frames=50):
    # 0=H, 1=A, 3=X. Confident runs are accepted retrospectively in their entirety.
    # Уверенную серию принимаем целиком после проверки её длины; метод офлайн.
    prediction=np.full(len(raw),3,dtype=np.int8); anchors=[]; gaps=[]
    if not len(raw):return prediction,anchors,gaps
    for a,b in runs_of(raw):
        if raw[a]!=3 and b-a>=min_frames:
            anchors.append((int(a),int(b),int(raw[a])))
            prediction[a:b]=raw[a]
    if not anchors:return prediction,anchors,gaps
    a,b,label=anchors[0];prediction[max(0,a-edge_frames):a]=label
    a,b,label=anchors[-1];prediction[b:min(len(raw),b+edge_frames)]=label
    for left,right in zip(anchors,anchors[1:]):
        _,start,old=left;stop,_,new=right
        if old==new:
            # The main rule: same confident team on both sides implies NO turnover.
            # Главное правило: одинаковые команды по краям означают НЕТ перехода.
            prediction[start:stop]=old
            boundary=None
        else:
            boundary=choose_boundary(evidence,start,stop,old,new,method)
            prediction[start:boundary]=old;prediction[boundary:stop]=new
        gaps.append(dict(start=start,stop=stop,old=old,new=new,boundary=boundary))
    return prediction,anchors,gaps

def predict_interval(cfg,first,stop,collect=False):
    # Crucially, train and validation calls cannot see each other's anchors/evidence.
    # Train и validation не видят якоря и свидетельства соседнего раздела.
    result=np.full(stop-first,3,dtype=np.int8);details=[];anchor_count=0;anchor_frames=0
    for half,hfirst,hlast in BLOCKS:
        a=max(first,hfirst);b=min(stop,hlast+1)
        if a>=b:continue
        states=dead[a:b]
        for local_start,local_stop in runs_of(states):
            x=a+local_start;y=a+local_stop
            if states[local_start]:result[x-first:y-first]=2;continue
            d=nearest[x:y];ok=reliable[x:y]
            raw=np.full(y-x,3,dtype=np.int8)
            for team in (0,1):
                confident=ok&(d[:,team]<=cfg['radius_m'])&(d[:,1-team]>=cfg['radius_m']+cfg['opponent_margin_m'])
                raw[confident]=team
            # Gaussian distance support; these scores are NOT calibrated probabilities.
            # Гауссовы оценки близости НЕ являются калиброванными вероятностями.
            evidence=np.exp(-.5*(d/max(cfg['radius_m'],1.))**2)
            evidence[~ok]=0.
            evidence=smooth_evidence(evidence,cfg['evidence_window_frames'])
            pred,anchors,gaps=fill_live(raw,evidence,cfg['min_confident_frames'],cfg['boundary_method'])
            result[x-first:y-first]=pred
            anchor_count+=len(anchors);anchor_frames+=sum(j-i for i,j,t in anchors)
            if collect:
                for gap in gaps:
                    details.append(dict(start_frame=x+gap['start'],end_frame_exclusive=x+gap['stop'],
                        left_team='HA'[gap['old']],right_team='HA'[gap['new']],
                        boundary_frame=None if gap['boundary'] is None else x+gap['boundary'],
                        rule='same_team_no_turnover' if gap['old']==gap['new'] else cfg['boundary_method']))
    return np.array(['H','A','D','X'])[result],dict(anchor_runs=anchor_count,anchor_frames=anchor_frames),details

# Behavioral tests: same-team bridge, exact confident labels, all missing, and changepoint.
# Проверки: одинаковая команда по краям, неизменность якорей, пропуски и граница.
for method in ('midpoint','weighted','consistent'):
    raw=np.array([1,1,3,3,3,1,1]);p,_,_=fill_live(raw,np.zeros((7,2)),2,method)
    assert np.all(p==1)
    raw=np.array([0,0,3,3,1,1]);p,_,_=fill_live(raw,np.zeros((6,2)),2,method)
    assert np.all(p[:2]==0) and np.all(p[-2:]==1)
    assert np.count_nonzero(p[1:]!=p[:-1])==1
    p,_,_=fill_live(np.full(7,3),np.zeros((7,2)),2,method)
    assert np.all(p==3)
e=np.array([[1.,0.],[1.,0.],[1.,0.],[0.,1.],[0.,1.],[0.,1.]])
assert choose_boundary(e,2,4,0,1,'consistent')==3
print('Anchor rules verified')


## Изолированный подбор: только первые четыре минуты

In [ ]:
gt_path=ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv'
truth_table=pm.read_label_csv(gt_path)
truth=pm.label_codes(truth_table)
public=np.flatnonzero(truth!=pm.X)
assert len(public)==7500 and np.all(np.diff(public)==1)
START,END=int(public[0]),int(public[-1]+1);SPLIT=START+240*FPS

def score(pred,first,stop):
    true=truth[first:stop];codes=pm.label_codes(pred)
    assert len(codes)==len(true)
    cm=pm.scored_confusion(pm.confusion(true,codes));runs=pm.valid_runs(true)
    spell=pm.spell_scores(true,pm.fill_abstentions(codes,runs),runs)
    return dict(macroF1=pm.macro_f1(cm),spellF1=spell['mean_f1'],
        accuracy=float(np.mean(true==codes)),X_fraction=float(np.mean(codes==pm.X)),
        true_possessions=spell['n_gt'],predicted_possessions=spell['n_pred'],
        **{name+'_F1':pm.per_class_prf(cm,k)[2] for k,name in enumerate(['H','A','D'])})

save_json(OUT/'run_metadata.json',dict(grid=GRID,train=[START,SPLIT-1],validation=[SPLIT,END-1],
    split_policy='isolated intervals; no future validation anchors in training',
    same_team_rule='always fill same-team anchored gaps unless separated by D',
    edge_extension_frames=50,dead_config=CFG,
    sources={k:sha256(PATHS[k]) for k in ['atd_raw','atd_meta','smart_tagging']},public_gt_sha256=sha256(gt_path)))
keys=list(GRID);configs=[dict(zip(keys,v)) for v in itertools.product(*(GRID[k] for k in keys))]
rows=[]
for i,cfg in enumerate(tqdm(configs,desc='Anchor grid / Перебор якорей')):
    pred,stats,_=predict_interval(cfg,START,SPLIT)
    rows.append(dict(candidate=i,**cfg,**score(pred,START,SPLIT),**stats))
    if i%10==0:
        save_json(OUT/'progress.json',dict(stage='grid',done=i+1,total=len(configs)))
        pd.DataFrame(rows).to_csv(OUT/'grid_train.csv',index=False)
table=pd.DataFrame(rows).sort_values(['spellF1','macroF1','candidate'],ascending=[False,False,True])
table.to_csv(OUT/'grid_train.csv',index=False)
selected=configs[int(table.iloc[0].candidate)]
save_json(OUT/'selected_config.json',selected)
print('Selected using training only:',selected)


## Проверка и полный прогноз

In [ ]:
# Inspect all three families only AFTER training-based selection. Never reselect on validation.
# Проверяем три семейства ПОСЛЕ выбора по обучению; по проверке победителя не меняем.
family_rows=[]
for method in GRID['boundary_method']:
    row=table[table.boundary_method==method].iloc[0]
    cfg=configs[int(row.candidate)]
    pred,stats,_=predict_interval(cfg,SPLIT,END)
    family_rows.append(dict(method=method,candidate=int(row.candidate),**score(pred,SPLIT,END),**stats))
pd.DataFrame(family_rows).to_csv(OUT/'boundary_methods_validation.csv',index=False)
train,train_stats,_=predict_interval(selected,START,SPLIT)
validation,val_stats,_=predict_interval(selected,SPLIT,END)
labels,full_stats,gaps=predict_interval(selected,0,N,collect=True)
result=pd.DataFrame({'frame':np.arange(N),'label':labels})
error=pm.validate_submission(result,N);assert error is None,error
result.to_csv(OUT/'submission.csv',index=False)
pd.DataFrame({'frame':np.arange(SPLIT,END),'label':validation}).to_csv(OUT/'validation_predictions.csv',index=False)
pd.DataFrame(gaps).to_csv(OUT/'gap_diagnostics.csv',index=False)
# This full-match retrospective result may use anchors AFTER the public window.
# Полный офлайн-прогноз может использовать якоря ПОСЛЕ публичного отрезка.
report=dict(train_isolated=score(train,START,SPLIT),validation_isolated=score(validation,SPLIT,END),
            public_full_context=score(labels[START:END],START,END),
            train_anchors=train_stats,validation_anchors=val_stats,full_anchors=full_stats)
save_json(OUT/'metrics.json',report)
comparison=[]
for name,path in [('001_box5_vote100',ROOT/'local_review/experiments/001_box5_vote100/box5_vote100.csv'),
                  ('004_mass_near_ball',ROOT/'local_review/experiments/004_mass_near_ball/submission.csv')]:
    if path.exists():
        prior=pm.read_label_csv(path);assert pm.validate_submission(prior,N) is None
        comparison.append(dict(experiment=name,context='prior causal history',**score(prior.label.to_numpy()[SPLIT:END],SPLIT,END)))
comparison.append(dict(experiment=EXPERIMENT,context='isolated offline minute',**report['validation_isolated']))
pd.DataFrame(comparison).to_csv(OUT/'comparison_validation.csv',index=False)
save_json(OUT/'progress.json',dict(stage='completed',grid_candidates=len(configs),
    seconds=time.perf_counter()-started,submission_sha256=sha256(OUT/'submission.csv')))
print(json.dumps(report,indent=2))
print(pd.DataFrame(family_rows).to_string(index=False))


## Проверка правил и сравнение контекста
Изолированная проверочная минута и полный офлайн-прогноз — разные условия. Показываем оба, не меняя выбранных параметров.

In [ ]:
# Verify every same-team gap and every D barrier in the exported answer.
# Проверяем все мосты одинаковой команды и все барьеры D в сохранённом ответе.
assert np.array_equal(labels=='D',dead)
for gap in gaps:
    a,b=gap['start_frame'],gap['end_frame_exclusive']
    assert not dead[a:b].any()
    if gap['left_team']==gap['right_team']:
        assert np.all(labels[a:b]==gap['left_team'])
    else:
        k=gap['boundary_frame']
        assert a<=k<=b
        assert np.all(labels[a:k]==gap['left_team'])
        assert np.all(labels[k:b]==gap['right_team'])
# Change ALL future geometry: training output must remain identical.
# Меняем всю будущую геометрию: результат обучающего отрезка не должен измениться.
saved=nearest[SPLIT:].copy()
try:
    nearest[SPLIT:]=np.inf
    isolated,_,_=predict_interval(selected,START,SPLIT)
    assert np.array_equal(isolated,train)
finally:
    nearest[SPLIT:]=saved
report['validation_full_context']=score(labels[SPLIT:END],SPLIT,END)
report['validation_context_changed_frames']=int(np.count_nonzero(labels[SPLIT:END]!=validation))
report['same_team_gaps_filled']=sum(g['left_team']==g['right_team'] for g in gaps)
report['different_team_gaps']=sum(g['left_team']!=g['right_team'] for g in gaps)
report['invariants_verified']=True
save_json(OUT/'metrics.json',report)
lines=['# 005 — обзор результата',
       '', 'Перебрано 243 конфигурации. Выбор по первым четырём минутам, без изменения после проверки.',
       'Выбранные параметры: '+json.dumps(selected,ensure_ascii=False),
       '', '| Условия | macroF1 | spellF1 | Доля X |', '|---|---:|---:|---:|']
for key in ['train_isolated','validation_isolated','validation_full_context','public_full_context']:
    r=report[key]
    lines.append(f"| {key} | {r['macroF1']:.4f} | {r['spellF1']:.4f} | {r['X_fraction']:.1%} |")
lines += ['', 'Изолированная проверка не получает якорей вне проверочной минуты. Полный прогноз использует будущие наблюдения всего тайма, но не будущие истинные метки.',
          'Оценка на всех пяти публичных минутах включает данные подбора параметров и не является независимым тестом. Проверочная минута ранее использовалась в проекте.',
          'Правило одинаковых команд по краям применяется всегда внутри живого интервала, независимо от длины. Возможные короткие перехваты и обратные отборы внутри такого интервала будут пропущены по конструкции.',
          'Все мосты, границы и барьеры D проверены. Изменение будущей геометрии не меняет изолированный обучающий прогноз.',
          'Детектор D оставлен прежним. Масштабный подбор на 14 обучающих владениях легко переобучается. Полезность метода на остальном матче без разметки неизвестна.',
          'Подробности: grid_train.csv, boundary_methods_validation.csv, gap_diagnostics.csv; итоговый файл: submission.csv.']
(OUT/'REVIEW_RU.md').write_text('\n'.join(lines)+'\n')
print('Context diagnostics:',json.dumps(report['validation_full_context']))
print('All anchor and split invariants passed')
